In [96]:
import pandas as pd
import torch
import time
import json
import os

In [ ]:
df = pd.read_csv('../booksummaries/book_data_clean.csv')
df.head()

,id,title,author,pub_date,genres,summary
0,620,Animal Farm,George Orwell,1945-08-17,"[""Roman à clef"", ""Satire"", ""Children's literat...","Old Major, the old boar on the Manor Farm, ca..."
1,843,A Clockwork Orange,Anthony Burgess,1962-01-01,"[""Science Fiction"", ""Novella"", ""Speculative fi...","Alex, a teenager living in near-future Englan..."
2,986,The Plague,Albert Camus,1947-01-01,"[""Existentialism"", ""Fiction"", ""Absurdist ficti...",The text of The Plague is divided into five p...
3,1756,An Enquiry Concerning Human Understanding,David Hume,NaN,[],The argument of the Enquiry proceeds by a ser...
4,2080,A Fire Upon the Deep,Vernor Vinge,NaN,"[""Hard science fiction"", ""Science Fiction"", ""S...",The novel posits that space around the Milky ...


In [98]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using GPU') if device == 'cuda' else print('Using CPU')

cols = [c for c in ['title', 'author', 'genres', 'summary'] if c in df.columns]
text = df[cols].fillna('').astype(str).agg(' | '.join, axis=1)

corpus = b'\n'.join(t.encode('utf-8') for t in text)

# print(list(corpus[:200]))

Using GPU


In [99]:
tokens = torch.tensor(list(corpus), dtype=torch.int32, device=device)

vocab = {i:bytes([i]) for i in range(256)}
merges= []
next_id = 256

special_tokens = ["[PAD]", "[CLS]", "[SEP]", "[BOS]", "[EOS]", "[QUERY]", "[BOOK]", "[SUM]"]
special_token_ids = {}
for tok in special_tokens:
    special_token_ids[tok] = next_id
    vocab[next_id] = tok.encode('utf-8')
    next_id += 1

vocab_size = 16000
min_freq = 2
max_merges = vocab_size - next_id 

In [100]:
@torch.no_grad()
def get_pair_counts(tokens: torch.Tensor):
    if tokens.numel() < 2:
        return None
    a = tokens[:-1].to(torch.int64)
    b = tokens[1:].to(torch.int64)
    pairs = (a<<32) | b
    sorted_pairs, _ = torch.sort(pairs)
    unique, counts = torch.unique_consecutive(sorted_pairs, return_counts=True)
    if unique.numel() == 0:
        return None
    
    best_idx = torch.argmax(counts)
    best_key = unique[best_idx].item()
    best_pair = ((best_key >> 32) & 0xFFFFFFFF, best_key & 0xFFFFFFFF)
    best_freq = counts[best_idx].item()
    return (best_pair, best_freq)

In [101]:
@torch.no_grad()
def merge(tokens:torch.Tensor, pair, new_id):
    a,b = pair
    if tokens.numel() < 2:
        return tokens
    first = tokens[:-1] == a
    second = tokens[1:] == b
    match = first & second
    if not torch.any(match):
        return tokens
    idx = torch.nonzero(match, as_tuple=False).squeeze(1)
    tokens[idx] = new_id

    mask = torch.ones(tokens.size(0), dtype=torch.bool, device=tokens.device)
    mask[idx + 1] = False
    return tokens[mask]

In [102]:
for i in range(max_merges):
    res = get_pair_counts(tokens)
    if res is None:
        break
    best_pair, best_freq = res
    before = tokens.numel()
    tokens = merge(tokens, best_pair, next_id)
    if tokens.numel() == before:
        break
    vocab[next_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
    merges.append((best_pair, next_id))
    next_id+=1
    if next_id >= vocab_size or tokens.numel() < 2:
        break

In [104]:
os.makedirs('bpe_model', exist_ok=True)
with open('bpe_model/merges.json', 'w', encoding='utf-8') as f:
    json.dump({'merges': [{'pair': p, 'id': i} for p, i in merges], 'special_tokens': special_token_ids}, f)
with open('bpe_model/vocab.json', 'w', encoding='utf-8') as f:
    json.dump({str(k): v.hex() for k,v in vocab.items()}, f)
print('Saved optimized model to bpe_model/.')

Saved optimized model to bpe_model/.
